# Run a larger multiscale ETW example

Change **only `N`** in the first code cell to any power of two at least 32, then use **Run → Run All Cells**.

The notebook performs the same fixed-radius coarse-to-fine run used for the presentation illustration. It writes:

- a four-panel slide preview;
- an all-levels figure;
- a JSON summary;
- a compressed NPZ containing every band and selected path; and
- one `*_RETURN_THIS.zip` file to send back for incorporation into the slide.

`COMPUTE_EXACT` is `False` by default so that large examples do not allocate the full finest grid. The multiscale run itself remains exact inside each displayed corridor.

In [ ]:
from pathlib import Path
import sys
import time

from IPython.display import HTML, Image, display

# Main input: use any power of two, e.g. 512, 1024, 2048, or 4096.
N = 1024

# These reproduce the current slide example, apart from its size.
SEED = 356
NOISE = 0.02
WARP_STRENGTH = 0.5
SIGMA = 0.25
COARSEST_SIZE = 32
RADIUS = 4

# Leave False for a large run.  True also runs the unrestricted dense solver.
COMPUTE_EXACT = False
WARM_UP_NUMBA = True

# Find the repository root whether the notebook server started in the root or
# in the examples directory.
ROOT = Path.cwd().resolve()
if not (ROOT / "hellinger_etw_multiscale.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "hellinger_etw_multiscale.py").exists():
    raise RuntimeError("Start Jupyter from the repository root, then reopen this notebook.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.multiscale_visualization import (
    record_fixed_radius_run,
    write_shareable_run_bundle,
)

if N < COARSEST_SIZE or N & (N - 1):
    raise ValueError(f"N={N} must be a power of two and at least {COARSEST_SIZE}.")
if COARSEST_SIZE & (COARSEST_SIZE - 1):
    raise ValueError("COARSEST_SIZE must be a power of two.")

print(f"Repository root: {ROOT}")
print(f"Finest grid: {N} x {N}")

In [ ]:
# Compile the Numba kernels on a tiny problem so compilation time is not
# mixed into the larger run below.
if WARM_UP_NUMBA:
    _ = record_fixed_radius_run(
        n=64,
        seed=SEED,
        noise=NOISE,
        warp_strength=WARP_STRENGTH,
        sigma=SIGMA,
        coarsest_size=COARSEST_SIZE,
        radius=RADIUS,
        use_numba=True,
        compute_exact=False,
    )
    print("Numba warm-up complete.")

In [ ]:
start = time.perf_counter()
trace = record_fixed_radius_run(
    n=N,
    seed=SEED,
    noise=NOISE,
    warp_strength=WARP_STRENGTH,
    sigma=SIGMA,
    coarsest_size=COARSEST_SIZE,
    radius=RADIUS,
    use_numba=True,
    compute_exact=COMPUTE_EXACT,
)
elapsed = time.perf_counter() - start

rows = trace.summary_rows()
header = "<tr><th>stage</th><th>grid</th><th>radius</th><th>evaluated cells</th><th>grid fraction</th><th>score</th></tr>"
body = "".join(
    "<tr>"
    f"<td>{row['stage']}</td>"
    f"<td>{row['shape']}</td>"
    f"<td>{row['radius']}</td>"
    f"<td>{row['allowed_cells']:,} / {row['full_cells']:,}</td>"
    f"<td>{100.0 * row['density']:.3f}%</td>"
    f"<td>{row['score']:.10f}</td>"
    "</tr>"
    for row in rows
)
display(HTML(f"<table>{header}{body}</table>"))
print(f"Run time after warm-up: {elapsed:.3f} s")
print(f"Total cells over all levels: {trace.total_allowed_cells:,}")
print(f"Finest full grid: {trace.finest_full_cells:,}")
print(f"Reduction factor: {trace.finest_full_cells / trace.total_allowed_cells:.2f}x")
if trace.exact_score is not None:
    print(f"Dense exact score gap: {trace.exact_score - trace.final_score:.6g}")

In [ ]:
output_dir = ROOT / "user_runs" / f"N_{N}_seed_{SEED}"
files = write_shareable_run_bundle(
    trace,
    output_dir=output_dir,
    prefix=f"multiscale_run_N{N}_seed{SEED}",
    max_slide_levels=4,
)

display(Image(filename=str(files["slide_levels_png"])))
print("\nFiles written:")
for label, path in files.items():
    print(f"  {label:18s} {path.relative_to(ROOT)}")
print("\nPlease send back this file:")
print(files["return_zip"])
